In [155]:
corpus = [
    "Dogs are loyal animals and are often called man's best friend.",
    "My dog loves to play fetch in the park every evening.",
    "My dog is a friendly dog and this dog loves to play with other dogs in the park.",
    "A dog needs care because every dog depends on its owner and a happy dog lives longer.",
    "The dog ran after another dog while the small dog barked loudly at the big dog.",
    "Many families adopt dogs from animal shelters.",
    "A dog can be trained to help people with disabilities.",
    "Puppies require proper nutrition and regular veterinary care.",
    "Some dogs are trained as police dogs to detect illegal substances.",
    "Walking the dog daily keeps both the pet and owner healthy.",
    "Dogs communicate using barking, body language, and tail wagging.",
    "Golden retrievers are popular family dogs due to their friendly nature.",
    "Training a dog requires patience, consistency, and rewards.",
    "Dogs have an excellent sense of smell compared to humans.",
    "Service dogs assist visually impaired individuals in daily life.",
    "Taking your dog to the vet regularly ensures good health.",
    "A dog’s diet should include balanced nutrients and clean water.",
    "Rescue dogs can become loving companions in a new home.",
    "Walking the dog daily improves health and mood.",
    "A child was excited to bring home a dog for companionship.",
    "The loyal pet waited by the door for its owner to return.",
    "Many families adopt a furry companion from animal shelters.",
    "The playful puppy enjoyed running across the green field.",
    "This animal has an excellent sense of smell and can be trained easily.",
    "Service animals assist people with disabilities in daily life."
    ]

In [156]:
import re
import nltk
nltk.download('wordnet', quiet=True)
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

STOPWORDS = {
    'i', 'me', 'my', 'a', 'an', 'the', 'and', 'or', 'but', 'are',
    'is', 'was', 'to', 'of', 'in', 'it', 'its', 'this', 'that',
    'for', 'on', 'at', 'be', 'by', 'as', 'up', 'do', 'so', 'if', 's'
}

def normalize_tex_bm25(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = [lemmatizer.lemmatize(t) for t in text.split() if t not in STOPWORDS]
    return " ".join(tokens)

In [157]:
clean_corpus = [normalize_tex_bm25(doc) for doc in corpus]
clean_corpus[1]

'dog love play fetch park every evening'

In [158]:
tokenized_doc = [doc.split() for doc in clean_corpus]
print(tokenized_doc[0])
print(tokenized_doc[0][0])

['dog', 'loyal', 'animal', 'often', 'called', 'man', 'best', 'friend']
dog


In [159]:
flattened_tokens = [w for doc in tokenized_doc for w in doc]

In [160]:
# Creating a list of unique words
unique_words = set()
for word in [w for doc in tokenized_doc for w in doc]:
    unique_words.add(word)
unique_words_list = list(unique_words)

In [161]:
def doc_len(document):
    document_clean = [normalize_tex_bm25(doc) for doc in document]
    document_tokenized = [term.split() for term in document_clean]
    document_flattened = [tokens for doc in document_tokenized for tokens in doc]
    document_length = len(document_flattened)
    return document_length

In [162]:
# Calculating term count
from collections import Counter

def term_count(word, tokens):
    return Counter(tokens).get(word, 0)

In [163]:
term_count('dog', flattened_tokens)

28

In [164]:
def term_frequency(word, document):
    doc_length = doc_len(document)
    return term_count(word, document)/doc_length

In [165]:
term_frequency('dogs', flattened_tokens)

0.0

In [166]:
# Calculate average document length
def avg_len(document):
    doc_clean = [normalize_tex_bm25(text) for text in document]
    doc_tokens = [token.split() for token in doc_clean]
    doc_mag = 0
    for tokens in doc_tokens:
        doc_mag += len(tokens)
    return doc_mag/len(doc_tokens)

In [167]:
avg_len(corpus)

7.88

In [168]:
import numpy as np
from collections import Counter

class BM25Index:
    def __init__(self, corpus, k=1.2, b=0.75, delta=1.0):
        self.k = k
        self.b = b
        self.delta = delta      # BM25+ lower bound
        self.corpus = corpus
        self.N = len(corpus)

        # Precompute tokenized docs once at index-build time
        self.tokenized_docs = [normalize_tex_bm25(doc).split() for doc in corpus]

        # Precompute doc lengths and avgdl
        self.doc_lengths = [len(doc) for doc in self.tokenized_docs]
        self.avgdl = sum(self.doc_lengths) / self.N

        # Build vocabulary — replaces global unique_words_list
        vocab = set(w for doc in self.tokenized_docs for w in doc)

        # Precompute IDF for every vocab term once — O(1) lookup during scoring
        self.idf_cache = {word: self._compute_idf(word) for word in vocab}

    def _compute_idf(self, word):
        """BM25-style IDF with +1 to prevent negative values when n_t > N/2."""
        doc_freq = sum(1 for doc in self.tokenized_docs if word in doc)
        return float(np.log((self.N - doc_freq + 0.5) / (doc_freq + 0.5) + 1))

    def _score_term(self, word, doc_idx):
        """BM25+ score for a single term in a single document.
        delta is only applied when the term is present, so documents
        without the term still score 0."""
        tf = Counter(self.tokenized_docs[doc_idx]).get(word, 0)
        if tf == 0:
            return 0.0
        dl = self.doc_lengths[doc_idx]
        tf_norm = tf * (self.k + 1) / (tf + self.k * (1 - self.b + self.b * (dl / self.avgdl)))
        idf = self.idf_cache.get(word, 0.0)   # 0 if word not in vocab
        return idf * (self.delta + tf_norm)

    def score(self, query):
        """Return BM25+ scores for all documents."""
        query_tokens = normalize_tex_bm25(query).split()
        return [
            sum(self._score_term(word, doc_idx) for word in query_tokens)
            for doc_idx in range(self.N)
        ]

    def rank(self, query, top_n=5):
        """Return top_n documents ranked by BM25+ score."""
        scores = self.score(query)
        ranked = sorted(zip(scores, range(self.N)), reverse=True)
        return [
            (round(score, 4), self.corpus[doc_idx])
            for score, doc_idx in ranked[:top_n]
            if score > 0
        ]

In [169]:
# Build the index once — tokenization, avgdl, and IDF are all precomputed here
bm25 = BM25Index(corpus)
print(f"Vocab size : {len(bm25.idf_cache)}")
print(f"Avg doc len: {bm25.avgdl:.2f}")
print(f"IDF('dog') : {bm25.idf_cache.get('dog', 0):.4f}")
print(f"IDF('loyal'): {bm25.idf_cache.get('loyal', 0):.4f}")

Vocab size : 127
Avg doc len: 7.88
IDF('dog') : 0.2877
IDF('loyal'): 2.3418


In [170]:
#   Calculate inverse document frequency
import numpy as np
def inverse_doc_freq(word, document):
    doc_clean_version = [normalize_tex_bm25(text) for text in document]
    doc_tokenized_version = [term.split() for term in doc_clean_version]
    num_doc = len(doc_tokenized_version)  # N
    doc_freq_with_words = 0  # n_t

    for doc in doc_tokenized_version:
        if word in doc:
            doc_freq_with_words += 1
    # BM25-style IDF with +1 inside log to prevent negative values when n_t > N/2
    idf_value = np.log((num_doc - doc_freq_with_words + 0.5) / (doc_freq_with_words + 0.5) + 1)
    return float(idf_value)

In [171]:
inverse_doc_freq('dogs', corpus)

3.9512437185814275

In [172]:
# Score all documents for a query
scores = bm25.score("I love my dog")
print("BM25+ scores:", scores)

BM25+ scores: [0.5735830359936334, 5.384855496067322, 5.204294571415173, 0.6942077007550148, 0.7341288013054262, 0.5891361590102959, 0.5891361590102959, 0.0, 0.6815579536730787, 0.5735830359936334, 0.5735830359936334, 0.5595560589336863, 0.6064788257133554, 0.5891361590102959, 0.5735830359936334, 0.5735830359936334, 0.5735830359936334, 0.5735830359936334, 0.6064788257133554, 0.6064788257133554, 0.0, 0.0, 0.0, 0.0, 0.0]


In [173]:
# Rank top results for a query
results = bm25.rank("I love my dog", top_n=5)
for rank, (score, doc) in enumerate(results, 1):
    print(f"Rank {rank} | score={score:.4f} | {doc}")

Rank 1 | score=5.3849 | My dog loves to play fetch in the park every evening.
Rank 2 | score=5.2043 | My dog is a friendly dog and this dog loves to play with other dogs in the park.
Rank 3 | score=0.7341 | The dog ran after another dog while the small dog barked loudly at the big dog.
Rank 4 | score=0.6942 | A dog needs care because every dog depends on its owner and a happy dog lives longer.
Rank 5 | score=0.6816 | Some dogs are trained as police dogs to detect illegal substances.


In [174]:
# Try a different query
results = bm25.rank("service animals disability", top_n=5)
for rank, (score, doc) in enumerate(results, 1):
    print(f"Rank {rank} | score={score:.4f} | {doc}")

Rank 1 | score=12.4353 | Service animals assist people with disabilities in daily life.
Rank 2 | score=4.7957 | A dog can be trained to help people with disabilities.
Rank 3 | score=4.6691 | Service dogs assist visually impaired individuals in daily life.
Rank 4 | score=3.1811 | Many families adopt dogs from animal shelters.
Rank 5 | score=3.0971 | This animal has an excellent sense of smell and can be trained easily.


In [175]:
# Stemming allows matching across word forms — 'dogs' and 'dog' both stem to 'dog'
results = bm25.rank("dogs park", top_n=5)
for rank, (score, doc) in enumerate(results, 1):
    print(f"Rank {rank} | score={score:.4f} | {doc}")

Rank 1 | score=5.3849 | My dog loves to play fetch in the park every evening.
Rank 2 | score=5.2043 | My dog is a friendly dog and this dog loves to play with other dogs in the park.
Rank 3 | score=0.7341 | The dog ran after another dog while the small dog barked loudly at the big dog.
Rank 4 | score=0.6942 | A dog needs care because every dog depends on its owner and a happy dog lives longer.
Rank 5 | score=0.6816 | Some dogs are trained as police dogs to detect illegal substances.


In [176]:
# Compare BM25 (delta=0) vs BM25+ (delta=1.0) on a query where term frequency is low
bm25_standard = BM25Index(corpus, delta=0.0)
bm25_plus     = BM25Index(corpus, delta=1.0)

query = "loyal dog"
print("=== BM25 (delta=0) ===")
for rank, (score, doc) in enumerate(bm25_standard.rank(query, top_n=3), 1):
    print(f"  Rank {rank} | score={score:.4f} | {doc}")

print("\n=== BM25+ (delta=1.0) ===")
for rank, (score, doc) in enumerate(bm25_plus.rank(query, top_n=3), 1):
    print(f"  Rank {rank} | score={score:.4f} | {doc}")

=== BM25 (delta=0) ===
  Rank 1 | score=2.6132 | Dogs are loyal animals and are often called man's best friend.
  Rank 2 | score=2.5951 | The loyal pet waited by the door for its owner to return.
  Rank 3 | score=0.4652 | My dog is a friendly dog and this dog loves to play with other dogs in the park.

=== BM25+ (delta=1.0) ===
  Rank 1 | score=5.2427 | Dogs are loyal animals and are often called man's best friend.
  Rank 2 | score=4.9369 | The loyal pet waited by the door for its owner to return.
  Rank 3 | score=0.7529 | My dog is a friendly dog and this dog loves to play with other dogs in the park.
